In [1]:
import numpy as np
from helpers import *
from implementations import *

## Import the data

In [2]:
# You have to change the path for it to work
data_path = r'C:\Users\natha\Documents\EPFL\Cours_MA1\ML\ML_course\projects\project1 - withGit\data\dataset'

In [3]:
x_train, x_test, y_train, train_ids, test_ids, headers_train = load_csv_data(data_path, sub_sample=False)

test


In [31]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(len(headers_train))

(328135, 321)
(109379, 321)
(328135,)
322


In [33]:
del headers_train[0]
print(len(headers_train))

321


## Convert -1 into 0 in y

In [34]:
y_train[y_train == -1] = 0

# Check if all values are either 0 or 1
if np.all((y_train == 0) | (y_train == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


In [35]:
x_copy = x_train.copy()
x_copy.shape

(328135, 321)

## Remove columns with too many Nan

In [53]:
# Function to find the count of missing values in each columns
def find_missing_values(data, headers=None):
    num_rows, num_cols = data.shape
    missing_count = np.zeros(num_cols, dtype=int)  
    columns = np.linspace(0,num_cols, num_cols+1)
    
    for col in range(num_cols):
        # Count the missing values
        missing_count[col] = np.sum(np.isnan(data[:, col]))
    if headers : 
        # Returning only the columns with missing values
        missing_info = {headers[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    else : 
        missing_info = {columns[col]: missing_count[col] for col in range(num_cols) if missing_count[col] > 0}
    return missing_info

We fix a threshold of 10%

In [54]:
def remove_high_missing_columns(data, headers=None, headers_to_keep=None, headers_to_remove=None):
    """
    Removes columns with high missing values from the dataset, with options to automatically keep or remove specific columns.

    Parameters:
    data (np.ndarray): The input NumPy array containing the data.
    headers (list of str): List of all column names in the dataset.
    headers_to_keep (list of str): List of column names to keep regardless of missing values.
    headers_to_remove (list of str): List of column names to remove regardless of missing values.

    Returns:
    np.ndarray: A filtered NumPy array with the remaining columns.
    list of str: The list of remaining headers.
    """
    assert data.shape[1] == len(headers)
    
    num_rows, num_cols = data.shape
    threshold = num_rows / 10

    # Find the missing values count for each column
    missing_count = find_missing_values(data, headers)
    
    # Determine the indices of columns to automatically keep and remove
    indices_to_keep = [headers.index(col) for col in headers_to_keep if col in headers] if headers_to_keep else []
    indices_to_remove = [headers.index(col) for col in headers_to_remove if col in headers] if headers_to_remove else []
    
    # Create the mask for columns that meet the criteria
    columns_to_keep = [
        col for col in range(num_cols)
        if (
            (headers[col] in headers_to_keep) or
            (headers[col] not in headers_to_remove and missing_count.get(headers[col], 0) <= threshold)
        )
    ]

    # Filter the data and headers to only include the columns that meet the criteria
    filtered_data = data[:, columns_to_keep]
    remaining_headers = [headers[col] for col in columns_to_keep]

    return filtered_data, remaining_headers

In [295]:
columns_to_keep = features_list = ['AGE', 'SEX', '_RACE', '_HISPANC', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'BPHIGH4', 'BLOODCHO', 'CHOLCHK', 'TOLDHI2',
                                   'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3', 'DIABETE3', 'DIABAGE2', 'DIABEDU', 'BLDSUGAR', 'INSULIN', 'PDIABTST', 'PREDIAB1',
                                   'DOCTDIAB', 'SMOKE100', 'SMOKDAY2', '_SMOKER3', 'STOPSMK2', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5', 'EXERANY2', '_FRUTSUM', 'METVL11',
                                   'FC60_', 'MINAC11', 'WEIGHT2', '_BMI5', 'HAVARTH3', 'BPMEDS']


columns_to_remove = ['Id', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_STATE', '_PSU', '_STSTR', 'HHADULT', 'CPDEMO1', 'EMPLOY1', 'CHILDREN', 
    'INTERNET', 'SEATBELT', 'IMFVPLAC', 'QSTVER', 'MSCODE', '_LLCPWT', '_STRWT', '_RAWRAKE', '_WT2RAKE', 'CLLCPWT', 'DUALCOR', 'WTKG3']

In [296]:
sliced_x_train, sliced_features = remove_high_missing_columns(x_copy, headers_train, columns_to_keep, columns_to_remove)

In [297]:
sliced_x_train.shape

(328135, 134)

In [298]:
print(sliced_features)
print(len(sliced_features))

['GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'CHOLCHK', 'TOLDHI2', 'CVDSTRK3', 'ASTHMA3', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD1', 'HAVARTH3', 'ADDEPEV2', 'CHCKIDNY', 'DIABETE3', 'DIABAGE2', 'SEX', 'MARITAL', 'EDUCA', 'RENTHOM1', 'VETERAN3', 'INCOME2', 'WEIGHT2', 'HEIGHT3', 'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SMOKE100', 'SMOKDAY2', 'STOPSMK2', 'USENOW3', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5', 'FRUITJU1', 'FRUIT1', 'FVBEANS', 'FVGREEN', 'FVORANG', 'VEGETAB1', 'EXERANY2', 'STRENGTH', 'FLUSHOT6', 'PNEUVAC3', 'HIVTST6', 'PDIABTST', 'PREDIAB1', 'INSULIN', 'BLDSUGAR', 'DOCTDIAB', 'DIABEDU', 'QSTLANG', '_DUALUSE', '_RFHLTH', '_HCVU651', '_RFHYPE5', '_CHOLCHK', '_LTASTH1', '_CASTHM1', '_ASTHMS1', '_DRDXAR1', '_PRACE1', '_MRACE1', '_HISPANC', '_RACE', '_RACEG21', '_RACEGR3', '_RACE_G1', '_AGEG5YR', '_AGE65YR', '_AGE80', '_AGE_G', 'HTIN4', 'HTM4', '_BMI5', '_BMI5CAT', '_RFBMI5', 

# Modify the data

### A function to check if the column has specific values

In [299]:
def unique_values(data, column_name, headers):
    """
    Returns a sorted list of unique values from a specified column, excluding NaN values.

    Parameters:
    data (np.ndarray): The input NumPy array containing the data.
    column_name (str): The name of the column to analyze.
    headers (list of str): List of all column names in the dataset.

    Returns:
    list: A sorted list of unique values from the specified column, excluding NaNs.
    """
    if column_name not in headers:
        raise ValueError(f"Column '{column_name}' not found in headers.")
    
    # Get the index of the specified column
    col_index = headers.index(column_name)
    
    # Extract the column values and filter out NaN values
    column = data[:, col_index]
    non_nan_values = column[~np.isnan(column)]
    
    # Get the unique values and sort them in ascending order
    unique_values = sorted(set(non_nan_values))
    
    return unique_values


In [300]:
print(unique_values(sliced_x_train, 'BLOODCHO', sliced_features))

[1.0, 2.0, 7.0, 9.0]


In [301]:
print(unique_values(sliced_x_train, 'HLTHPLN1', sliced_features))

[1.0, 2.0, 7.0, 9.0]


### A function to replace the values

In [302]:
x_reference = np.copy(sliced_x_train)

In [303]:
def replace_specific_values(data, headers):
    """
    Iterates through each column in the dataset and performs an action if the unique values of the column match the target values.

    Parameters:
    data (np.ndarray): The input NumPy array containing the data.
    headers (list of str): List of all column names in the dataset.

    Returns:
    None
    """
    accessed_columns = []

    for column_name in headers:
        # Get unique values from the column
        unique_vals = unique_values(data, column_name, headers)

        if unique_vals == [1, 2, 9]:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 0, column)
            column = np.where(column == 9, np.nan, column)
            
            # Update the data with modified column
            data[:, col_idx] = column
            

        if unique_vals == [1, 2, 7, 9]:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 0, column)
            column = np.where(column == 7, np.nan, column)
            column = np.where(column == 9, np.nan, column)
            
            # Update the data with modified column
            data[:, col_idx] = column
            

        if unique_vals == [1, 2, 3, 9]:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 1, column)
            column = np.where(column == 3, 0, column)
            column = np.where(column == 9, np.nan, column)
            
            # Update the data with modified column
            data[:, col_idx] = column
             
        
        if unique_vals == [1, 2, 3, 7, 9]:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 0, column)
            column = np.where(column == 3, 0, column)
            column = np.where(column == 7, np.nan, column)
            column = np.where(column == 9, np.nan, column)
            
            # Update the data with modified column
            data[:, col_idx] = column

        if unique_vals == [1, 2, 3, 4, 9] and column_name != '_PACAT1':
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 1, column)
            column = np.where(column == 3, 1, column)
            column = np.where(column == 4, 0, column)
            column = np.where(column == 9, np.nan, column)
            
            # Update the data with modified column
            data[:, col_idx] = column

        if unique_vals == [1, 2, 3, 4, 7, 9]:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, 0, column)
            column = np.where(column == 3, 0, column)
            column = np.where(column == 4, 0, column)
            column = np.where(column == 7, np.nan, column)
            column = np.where(column == 9, np.nan, column)
            
            # Update the data with modified column
            data[:, col_idx] = column

        if unique_vals == [1, 2, 3]:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 3, np.nan, column)
            
            # Update the data with modified column
            data[:, col_idx] = column
        
        if column_name in ['ALCDAY5', 'EXERHMM1', 'BLDSUGAR']:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 888, 0, column)
            column = np.where(column == 777, np.nan, column)
            column = np.where(column == 999, np.nan, column)

            # Update the data with modified column
            data[:, col_idx] = column

        if column_name in ['AVEDRNK2', 'INCOME2', 'LASTSMK2', 'MAXDRNKS', 'JOINPAIN']:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 77, np.nan, column)
            column = np.where(column == 99, np.nan, column)

            # Update the data with modified column
            data[:, col_idx] = column

        if column_name in ['DRNK3GE5', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'DOCTDIAB']:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 88, 0, column)
            column = np.where(column == 77, np.nan, column)
            column = np.where(column == 99, np.nan, column)

            # Update the data with modified column
            data[:, col_idx] = column

        if column_name == '_PACAT1':
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 9, np.nan, column)

            # Update the data with modified column
            data[:, col_idx] = column

        if column_name in ['FC60_', '_DRNKWEK', 'MAXVO2_', 'STRFREQ_']:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 88, 0, column)
            column = np.where(column == 77, np.nan, column)
            column = np.where(column == 99900, np.nan, column)

            # Update the data with modified column
            data[:, col_idx] = column

        if column_name in ['EDUCA', '_CHLDCNT', '_EDUCAG', 'SMOKER3', '_INCOMG', 'PAMISS1_']:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 9, np.nan, column)

            # Update the data with modified column
            data[:, col_idx] = column

        if column_name in ['_FRUITEX', '_VEGETEX']:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 2, np.nan, column)

            # Update the data with modified column
            data[:, col_idx] = column

        if column_name == '_AGEG5YR':
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 14, np.nan, column)

            # Update the data with modified column
            data[:, col_idx] = column
        
        if column_name == 'DROCDY3_':
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column == 900, np.nan, column)

            # Update the data with modified column
            data[:, col_idx] = column

        if column_name == 'MARITAL':
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column != 1, 0, column)

            # Update the data with modified column
            data[:, col_idx] = column

        if column_name == 'WEIGHT2':
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where((column >= 9000) & (column // 1000 == 9), np.nan, column)

            # Update the data with modified column
            data[:, col_idx] = column

        if column_name in ['FRUITJU1', 'FRUIT1', 'FVBEANS', 'FVORANG', 'VEGETAB1']:
            # Find the column index
            col_idx = headers.index(column_name)
            column = data[:, col_idx]

            column = np.where(column >= 300, 1, 0)

            # Update the data with modified column
            data[:, col_idx] = column

In [304]:
def get_unmodified_headers(old_data, new_data, headers):
    """
    Compares the old dataset with the new dataset and returns the headers 
    that have not been modified.

    Parameters:
    old_data (np.ndarray): The original NumPy array containing the old data.
    new_data (np.ndarray): The modified NumPy array containing the new data.
    headers (list of str): List of all column names in the dataset.

    Returns:
    list: A list of headers that have not been modified.
    """
    # Check the types of old_data and new_data
    if not isinstance(old_data, np.ndarray):
        raise ValueError("old_data must be a numpy ndarray")
    if not isinstance(new_data, np.ndarray):
        raise ValueError("new_data must be a numpy ndarray")
    
    if old_data.shape[1] != new_data.shape[1]:
        raise ValueError("old_data and new_data must have the same number of columns")
    
    unmodified_headers = []

    for i, column_name in enumerate(headers):
        # Check if the column in the old data is the same as in the new data
        if np.array_equal(old_data[:, i], new_data[:, i]):
            unmodified_headers.append(column_name)

    return unmodified_headers

# Example usage
# Make sure x_reference and sliced_x_train are defined as NumPy arrays
# Example (uncomment and replace with your actual datasets):
# x_reference = np


In [305]:
replace_specific_values(sliced_x_train, sliced_features)

In [306]:
intact_columns = get_unmodified_headers(x_reference, sliced_x_train, sliced_features)
print(intact_columns)
print(len(intact_columns))

['SEX', '_PRACE1', '_MRACE1', '_RACE', '_RACEGR3', '_AGE80', '_AGE_G', '_MISFRTN', '_MISVEGN', '_FRTRESP', '_VEGRESP', '_FRT16', '_VEG23', 'MAXVO2_', 'FC60_']
15


Should not be accessed indeed
['SEX', '_AGE80', '_AGE_G', 'MISFRTN', '_MISVEGN', '_FRTRESP', '_VEGRESP', '_FRT16', '_VEG23', ]

MAXVO2, FC60 check manually if still has 99900

In [307]:
np.any(sliced_x_train == 99900)

False

In [308]:
print(unique_values(sliced_x_train, 'BLOODCHO', sliced_features))
print(unique_values(sliced_x_train, '_PACAT1', sliced_features))
print(unique_values(sliced_x_train, 'HLTHPLN1', sliced_features))
print(unique_values(sliced_x_train, 'MENTHLTH', sliced_features))

[0.0, 1.0]
[1.0, 2.0, 3.0, 4.0]
[0.0, 1.0]
[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0]


## Set of rules to improve the dataset: as of now, many values don't make sense

I have read through all the features description in order to select those that were crucial (to keep even if many Nan), those that were meaningless (like the date), and to note down how to convert values that often make no sense into some that can be nicely handled.

In [ ]:
#Some sets have 1,2,3,4,5,6,7,9 should replace the 9

#BLDSUGAR if starts by 1, two last*365. If starts by 2, two last*52. If starts by 3, two last*12, if starts by 4, two last. If > 499, 0.

#EXERHMM1 if >= 100, first number* 60 + last two numbers


#_PRACE1 MAKE THE SPLIT
#_MRACE1 MAKE THE SPLI
#_RACE MAKE THE SPLIT
#_RACEGR3 MAKE THE SPLIT
#_RACE_G1 MAKE THE SPLIT

In [ ]:
# list of meaningless to add in function. Also take out redundant (ex: continous and discontinuous value of age...)
'HHADULT', 'CPDEMO1', 'EMPLOY1', 'CHILDREN', 'INTERNET', 'SEATBELT', 'IMFVPLAC', 'QSTVER', 'MSCODE', '_STRWT', 'RAWRAKE', 'CLLCPWT', 'DUALCOR', 'WTKG3'

# list of those that could be meaningful but not currently kept (is making sacrifice taking data that has a lot missing. Only make a few exceptions. And some redundant)
'BLOODCHO', 'CHOLCHK', 'CVDINFR4', 'CVDCRHD4', 'ASTHMA3', 'DIABETE3', 'SMOKE100', 'SMOKDAY2', 'STOPSMK2', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5', 'EXERANY2', 'INSULIN'
... 'PDIABTST', 'PREDIAB1', 'BLDSUGAR', 'DOCTDIAB', '_FRUTSUM', 'METVL11', 'FC60_', 'MINAC11'

# Modifs for them if keep them

#DIABETE3 replace 2 by 1, replace 3, 4, 7, 9 by 0
#SMOKDAY2 replace 2 by 1, replace 3, 7, 9 by 0
#USENOW3 replace 2 by 1, replace 3, 7, 9 by 0
#ALCDAY5 replace 777, 888, 999 by 0
#AVEDRNK replace 77, 99 by 0
#DRNK3GE5 replace 77, 88, 99 by 0
#PREDIAB1 replace 2 by 1, replace 3, 7, 9 by 0
#BLDSUGAR if starts by 1, two last*365. If starts by 2, two last*52. If starts by 3, two last*12, if starts by 4, two last. If > 499, 0.
#DOCTDIAB if >76, replace by 0
#FC60_ replace 99900 by average of rest

# General rule: try to reduce to binary. Keep as it is for kinda continuous data "ex: int from 1 to 9"

#MARITAL if not 1, replace by 0
#PHYSHLTH replace 88, 77, 99 by 0
#POORHLTH replace 88, 77, 99 by 0
#PERSDOC2 replace 2 by 1, replace 3, 7, 9 by 0
#SEX replace 2 by 0
#EDUCA replace 9 by 0
#INCOME2 replace 77 and 99 by 4
#WEIGHT2 if starts by 9, replace by 162
#LASTSMK2 replace 77, 99 by Nan
#MAXDRNKS replace 77, 99 by 0
#FRUITJU1 if >= 300 replace by 0, if < 300 replace by 1
#FRUIT1 if >= 300 replace by 0, if < 300 replace by 1
#FVBEANS if >= 300 replace by 0, if < 300 replace by 1
#FVORANG if >= 300 replace by 0, if < 300 replace by 1
#VEGETAB1 if >= 300 replace by 0, if < 300 replace by 1
#EXERHMM1 replace by 1, 2, 3, 4 if 1<30<100<200
#JOINPAIN replace 77 and 99 by 0
#_ASTHMS1 replace 2 by 1, replace 3, 9 by 0
#_PRACE1 MAKE THE SPLIT
#_MRACE1 MAKE THE SPLI
#_RACE MAKE THE SPLIT
#_RACEGR3 MAKE THE SPLIT
#_RACE_G1 MAKE THE SPLIT
#_AGEG5YR replace 14 by 4
#_AGE65YR replace 2, 3 by 0
#HTIN4 replace blank by 67
#_BMI5 replace Nan by average of rest
#_CHLDCNT replace 9 by 0
#_EDUCAG replace 9 by 1
#_SMOKER3 replace 9 by 4
#DROCDY3_ replace 900 by 0
#_DRNKWEK replace 99900 by 0
#_FRUITEX replace 2 by 1
#_VEGETEX replace 2 by 1
#MAXVO2_ replace 99900 by average of rest
#STRFREQ_ replace 99900 by average of rest
#_PACAT1 replace 9 by 4 (or average of rest)
#_PA150R2 replace 2, 3 and 9 by 0
#_PA300R2 replace 2, 3 and 9 by 0
#_PAREC1 replace 2 and 3 by 1, replace 4 and 9 by 0
#_PASTAE1 replace 2 and 9 by 0
#_LMTACT1 replace 9 by 3
#_LMTWRK1 replace 9 by 3
#_LMTWRK1 replace 9 by 4
#_RFSEAT2 replace 9 by 2
#_PNEUMO2 replace 9 by 0

## Standardize ? only when value is non binary

In [319]:
def standardize(x):
    """Standardize the input data x

    Args:
        x: numpy array of shape=(num_samples, num_features)

    Returns:
        standardized data, shape=(num_samples, num_features)
    """
    # Calculate the mean and standard deviation, ignoring NaN values
    means = np.nanmean(x, axis=0)
    stds = np.nanstd(x, axis=0)

    # Handle case where standard deviation is zero to avoid division by zero
    stds = np.where(stds == 0, 1, stds)  # Replace 0 std with 1 to avoid division by zero

    # Standardize the data
    std_data = (x - means) / stds

    return std_data

In [320]:
print(sliced_x_train[0,:])

[2.000e+00 1.000e+00 5.000e+00 0.000e+00 1.000e+00 1.000e+00 0.000e+00
 1.000e+00 0.000e+00       nan 1.000e+00 1.000e+00 0.000e+00 0.000e+00
 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 1.000e+00 0.000e+00
 0.000e+00       nan 2.000e+00 1.000e+00 5.000e+00 1.000e+00 0.000e+00
 8.000e+00 1.100e+02 5.010e+02 1.000e+00 0.000e+00 0.000e+00 0.000e+00
 0.000e+00 0.000e+00 0.000e+00 1.000e+00 1.000e+00 1.000e+00 0.000e+00
 0.000e+00       nan       nan 1.000e+00 0.000e+00 1.000e+00 3.030e+02
 1.000e+00 0.000e+00 1.000e+00 1.050e+02 1.000e+00 0.000e+00 0.000e+00
       nan       nan       nan       nan       nan       nan 1.000e+00
 0.000e+00 1.000e+00 1.000e+00 1.000e+00 1.000e+00 1.000e+00 1.000e+00
 0.000e+00 2.000e+00 1.000e+00 1.000e+00 0.000e+00 1.000e+00 1.000e+00
 1.000e+00 1.000e+00 8.000e+00 1.000e+00 5.700e+01 5.000e+00 6.100e+01
 1.550e+00 2.078e+01 2.000e+00 1.000e+00 1.000e+00 1.000e+00 5.000e+00
 1.000e+00 0.000e+00 0.000e+00 0.000e+00 1.000e+00 0.000e+00 1.000e+00
 0.000

In [321]:
x_standardized = standardize(sliced_x_train)
print(x_standardized[0,:])

[-0.51316305 -0.37180494  0.22436556 -0.5593509   0.28000565  0.53566424
 -0.32970375 -0.4607842  -0.82125739         nan  0.36005526  0.53326516
 -0.85340967 -0.20630691 -0.39410744 -0.32381417 -0.33118542 -0.29524542
 -0.71103771  2.0653609  -0.19075941 -0.38461167         nan  0.85611713
  0.94395446  0.08825436  0.61952836 -0.38842175  1.00662618 -0.20373686
 -0.17364068  1.74369426 -0.36297528 -0.22704168 -0.32796808 -0.45743527
 -0.21360044 -0.2907099   1.14171469  1.79098277  0.87340863 -0.13445763
 -0.94524478         nan         nan  0.76777465 -0.65906723  0.85130525
  0.31535355  0.91587499 -0.71835524  0.60077687 -1.36177446  1.03683732
 -0.87347065 -0.65018517         nan         nan         nan         nan
         nan         nan -0.19623869 -1.70516688  0.47547434  0.34168723
  0.82125739  0.36281234  0.39410744  0.31581494 -0.38801186  0.71103771
 -0.1678059  -0.17437949 -0.29777193 -0.44760941  0.53756221 -0.44450071
 -0.47519066  0.08285679 -0.7305528   0.09724365  0

## Deal with missing data (in development)

Input missing data by 0

In [322]:
x_imputed = np.nan_to_num(x_standardized, nan=0.0)
x_imputed.shape

(328135, 134)

In [323]:
column_with_nan = find_missing_values(x_imputed)

In [324]:
print(column_with_nan)

{}


# TRAINING THE MODEL

## Split the data

In [325]:
def split_data(x, y, ratio, seed=1):
    """
    split the dataset based on the split ratio. If ratio is 0.8
    you will have 80% of your data set dedicated to training
    and the rest dedicated to testing. If ratio times the number of samples is not round
    you can use np.floor. Also check the documentation for np.random.permutation,
    it could be useful.

    Args:
        x: numpy array of shape (N,), N is the number of samples.
        y: numpy array of shape (N,).
        ratio: scalar in [0,1]
        seed: integer.

    Returns:
        x_tr: numpy array containing the train data.
        x_te: numpy array containing the test data.
        y_tr: numpy array containing the train labels.
        y_te: numpy array containing the test labels.
    """
    
    # set seed
    np.random.seed(seed)
    # ***************************************************
    indices = np.random.permutation(x.shape[0])
    x_shuffled = x[indices]
    y_shuffled = y[indices]
    
    first_index_te = int(np.floor(x.shape[0] * ratio))
    x_tr = x_shuffled[:first_index_te]
    x_te = x_shuffled[first_index_te:]
    y_tr = y_shuffled[:first_index_te]
    y_te = y_shuffled[first_index_te:]
    return x_tr, x_te, y_tr, y_te
    # ***************************************************

In [326]:
x_tr, x_val, y_tr, y_val = split_data(x_imputed, y_train, 0.8, seed=1)
print(x_tr.shape)
print(x_val.shape)
print(y_tr.shape)
print(y_val.shape)

(262508, 134)
(65627, 134)
(262508,)
(65627,)


## Train the model

In [327]:
lambda_ = 0.1
initial_w = np.ones(x_tr.shape[1])*0.5
max_iters = 1000
gamma = 0.01

w_opt, loss_opt = reg_logistic_regression(y_tr, x_tr, lambda_, initial_w, max_iters, gamma)

Current iteration=0, loss=2.7154785320805765
Current iteration=100, loss=1.8828152052058231
Current iteration=200, loss=1.3224369962132474
Current iteration=300, loss=0.98810678367016
Current iteration=400, loss=0.8129577872590601
Current iteration=500, loss=0.7315921182583821
Current iteration=600, loss=0.6964603323375673
Current iteration=700, loss=0.681150255302132
Current iteration=800, loss=0.6740785703652286
Current iteration=900, loss=0.6705994399687989
loss=0.6688004966829698


In [328]:
print(w_opt)

[ 7.52358971e-02  2.20878779e-02  9.61837511e-04  2.25458631e-02
  2.95496027e-03 -3.45419754e-03  2.29830800e-02  2.94391425e-02
  9.64287668e-02  5.53715813e-02  1.11758602e-02  3.06841609e-02
  6.14225722e-02  9.77248350e-02  5.90955978e-02  2.08267388e-02
  1.07567674e-02  6.78461294e-02  3.27709428e-02  3.02079774e-03
  5.11592564e-02  5.38887386e-02  6.13836987e-02  8.41743517e-03
 -4.98263046e-05  3.76471693e-02 -5.10914371e-03  3.98851713e-02
  3.73174151e-04  7.36608705e-03 -1.28327143e-03  8.88377702e-03
  2.91038901e-02  2.91478558e-02  6.72432202e-03  2.92109812e-02
  1.01678896e-02  9.95298096e-03  2.59863449e-02  7.06286138e-02
  5.88883472e-02  1.33295904e-02 -1.83486795e-03  4.79118047e-02
  4.76463067e-02  1.80890429e-02  9.17777946e-03  7.54741034e-03
  1.82788244e-02  5.89659968e-03 -9.46691091e-03  9.92075224e-03
  5.92239130e-02  9.79451453e-03  2.95929181e-02 -6.34401343e-04
  4.17486510e-02  4.60242265e-02  6.71840679e-02  5.98589490e-02
  6.52638918e-02  5.84819

In [334]:
q = x_tr @ w_opt

limit = 0.5 #HYPERPARAM

q[q <= limit] = 0
q[q > limit] = 1

# Check if all values are either 0 or 1
if np.all((q == 0) | (q == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

accuracy = np.mean(y_tr == q)
print("Accuracy:", accuracy)

TP = np.sum((y_tr == 1) & (q == 1))
FP = np.sum((y_tr == 0) & (q == 1))
FN = np.sum((y_tr == 1) & (q == 0))

# Calculate Precision and Recall
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

# Calculate F1 Score
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)

All values are either 0 or 1.
Accuracy: 0.867051670806223
Precision: 0.3306343174510377
Recall: 0.48520075497597803
F1 Score: 0.39327561628594276


## Apply the model to the validation set

In [335]:
g = x_val @ w_opt
g.shape

(65627,)

In [336]:
limit = 0.3

g[g <= limit] = 0
g[g > limit] = 1

# Check if all values are either 0 or 1
if np.all((g == 0) | (g == 1)):
    print("All values are either 0 or 1.")
else:
    raise ValueError("The array contains values other than 0 or 1.")

All values are either 0 or 1.


## Check our accuracy and F1

In [337]:
accuracy = np.mean(y_val == g)

print("Accuracy:", accuracy)

Accuracy: 0.8099410303685983


In [338]:
TP = np.sum((y_val == 1) & (g == 1))
FP = np.sum((y_val == 0) & (g == 1))
FN = np.sum((y_val == 1) & (g == 0))

# Calculate Precision and Recall
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0

# Calculate F1 Score
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1_score)

Precision: 0.2615546218487395
Recall: 0.6595444110895285
F1 Score: 0.3745675174246603
